# 使用 PROC GVARCLUS 降低产线传感器冗余度

## 执行摘要

一条多区域生产线实时传输数十个相互关联的传感器读数,其中许多携带着相同的底层信号。本笔记本使用 **PROC GVARCLUS**(变量聚类)将 13 个工艺传感器分成四个不相交的聚类,然后读取每个聚类的 R 方值,为每个聚类选出一个代表性测量点——在不损失工艺信息的前提下削减统计过程控制(SPC)监控的规模。四个聚类中有三个清晰地对应到物理区域(平均 R 方值分别为 0.92、0.93 和 0.96);第四个是该过程拆分出的单通道聚类,我们对此进行了深入探讨,而非略过不谈。

## 数据来源

所有数据均通过 `call streaminit(20260531)` 与 `rand()` 在程序内生成——不涉及任何外部或网络输入。

| 数据集 | 行数 | 关键变量 | 说明 |
|---------|------|---------------|-------------|
| `process_sensors` | 100 | `zone1_a`–`zone1_c`, `zone2_a`–`zone2_c`, `zone3_a`, `zone3_b`, `temp_in`, `temp_out`, `pressure_a`, `pressure_b`, `vibration` | 来自三区生产线的合成读数。同一区域内的传感器共享一个潜在驱动信号(高度相关);跨仪表通道(与1区相关的温度、与2区相关的压力、与3区相关的振动)被加入以营造真实的冗余。DATA 步循环请求 300 个周期,但本评估环境以无许可模式运行,只保留前 100 条观测——足以清晰地还原出聚类结构。 |
| `clusters`(OUT=) | 13 | `Variable`, `Cluster`, `RSq_Own` | 每个输入传感器一行:其被分配到的聚类及其与该聚类分量的 R 方值。由 `OUTPUT OUT=` 语句生成。 |

# 使用 PROC GVARCLUS 降低产线传感器冗余度

在装有仪表的生产线上,记录数十个测量相互重叠的物理现象的传感器是很常见的情况——同一区域内有多个热电偶、冗余的压力变送器、追踪同一根轴的振动探头。把每一个通道都输入控制图或下游模型,既浪费监控精力,又会加剧多重共线性。

**变量聚类**正是针对这一问题的直接解法。`PROC GVARCLUS` 将数值变量划分为*互不相交*的聚类,每个聚类由其成员的第一主成分来概括。变动趋势一致的传感器会落入同一聚类;工程师随后便可为每个聚类只保留一个代表性通道。

在本笔记本中,我们将:

1. 模拟一条三区生产线的读数,并刻意引入冗余传感器(循环请求 300 个周期,此处保留 100 个)。
2. 使用 `PROC GVARCLUS` 在 `MAXCLUSTERS=4` 下对 13 个通道进行聚类。
3. 将聚类分配结果记录到输出数据集中并加以汇总。
4. 解读 R 方值,为持续 SPC 监控挑选每个聚类的代表测量点。

## 步骤 1 —— 生成合成传感器数据

我们模拟三个工艺区域,每个区域都有一个隐藏的驱动信号(`zone1_a`、`zone2_a`、`zone3_a`)。其余通道是各自区域驱动信号的带噪声线性函数,因此同一区域内高度相关,而跨区域之间则几乎独立。我们还将进/出口温度与1区绑定,将两个压力变送器与2区绑定,以模拟真实产线上常见的跨仪表冗余。固定的随机种子使运行结果可复现。循环请求 300 个周期;在无许可模式下,引擎只保留前 100 条观测,下方的 NOTE 会报告这一点。

In [1]:
数据 process_sensors;
    调用 streaminit(20260531);
    循环 cycle = 1 到 300;
        /* 1区:潜在驱动信号加两个冗余探头 */
        zone1_a = rand("normal", 0, 1);
        zone1_b = 0.90*zone1_a + rand("normal", 0, 0.30);
        zone1_c = 0.85*zone1_a + rand("normal", 0, 0.35);

        /* 2区:潜在驱动信号加两个冗余探头 */
        zone2_a = rand("normal", 0, 1);
        zone2_b = 0.88*zone2_a + rand("normal", 0, 0.30);
        zone2_c = 0.82*zone2_a + rand("normal", 0, 0.40);

        /* 3区:潜在驱动信号加一个冗余探头 */
        zone3_a = rand("normal", 0, 1);
        zone3_b = 0.90*zone3_a + rand("normal", 0, 0.30);

        /* 与现有驱动信号绑定的跨仪表通道 */
        temp_in    = 180 + 4.0*zone1_a + rand("normal", 0, 1.5);
        temp_out   = 178 + 4.0*zone1_a + rand("normal", 0, 1.6);
        pressure_a = 2.5 + 0.60*zone2_a + rand("normal", 0, 0.20);
        pressure_b = 2.5 + 0.58*zone2_a + rand("normal", 0, 0.22);
        vibration  = 0.40 + 0.30*zone3_a + rand("normal", 0, 0.10);
        输出;
    结束;
    删除 cycle;
运行;


NOTE: DATA process_sensors

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote process_sensors (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.07 seconds
  cpu   0.07 seconds


## 步骤 2 —— 对传感器进行聚类

我们对全部 13 个通道调用 `PROC GVARCLUS`。`VAR` 语句列出要参与聚类的数值型传感器。我们用 `MAXCLUSTERS=4` 将划分数量上限设为四(预计大致每个物理区域对应一个聚类,并留有少量余地)。`ODS GRAPHICS ON` 配合 `PLOTS=ALL` 会启用变量聚类树状图;引擎会将该 SVG 写入工作目录而不是内嵌渲染,因此我们下面解读的结构来自打印出的 **Variable Cluster Assignments**(变量聚类分配)表和各聚类的特征值表。

`OUTPUT OUT=` 语句会将变量到聚类的分配结果——连同每个变量相对其自身聚类的 R 方值——写入一个数据集,供我们之后继续处理。

In [2]:
ODS GRAPHICS ON;

过程 gvarclus 数据=process_sensors maxclusters=4 PLOTS=ALL;
    变量 zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
    标签 zone1_a = "1区传感器A"
          zone1_b = "1区传感器B"
          zone1_c = "1区传感器C"
          zone2_a = "2区传感器A"
          zone2_b = "2区传感器B"
          zone2_c = "2区传感器C"
          zone3_a = "3区传感器A"
          zone3_b = "3区传感器B"
          temp_in = "进口温度"
          temp_out = "出口温度"
          pressure_a = "压力变送器A"
          pressure_b = "压力变送器B"
          vibration = "振动";
    输出 out=clusters;
运行;

ODS GRAPHICS OFF;


                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  1区传感器A                                  1     0.968670
  1区传感器B                                  1     0.918900
  1区传感器C                                  4     1.000000
  2区传感器A                                  2     0.983640
  2区传感器B                                  2     0.946708
  2区传感器C                                  2     0.892405
  3区传感器A                                  3     0.977237
  3区传感器B                                  3     0.949054
  进口温度                                    1     0.903394
  出口温度                                    1     0.889519
  压力变送器A                                  2     0.929356
  压力变送器B                                  2     0.920915
  振动         


NOTE: ODS Graphics is ON (width=640px, height=480px, format=SVG).
NOTE: PROC GVARCLUS data=process_sensors

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.
NOTE: ODS Graphics is OFF.


## 步骤 3 —— 使用 SUMMARY 选项重新运行

第二次运行该过程并加上 `SUMMARY` 选项,得到的划分保持不变。这一遍用 `PLOTS=NONE` 关闭图形输出。在本环境中,打印出的报告与步骤 2 完全相同——同样的 13 行分配表、同样的四个聚类、同样的特征值汇总——因此该单元格主要用于演示 `SUMMARY` 和 `PLOTS=NONE` 选项能够正确解析并运行;并不期望产生新的数字。

In [3]:
过程 gvarclus 数据=process_sensors maxclusters=4 summary PLOTS=none;
    变量 zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
    标签 zone1_a = "1区传感器A"
          zone1_b = "1区传感器B"
          zone1_c = "1区传感器C"
          zone2_a = "2区传感器A"
          zone2_b = "2区传感器B"
          zone2_c = "2区传感器C"
          zone3_a = "3区传感器A"
          zone3_b = "3区传感器B"
          temp_in = "进口温度"
          temp_out = "出口温度"
          pressure_a = "压力变送器A"
          pressure_b = "压力变送器B"
          vibration = "振动";
运行;


                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  1区传感器A                                  1     0.968670
  1区传感器B                                  1     0.918900
  1区传感器C                                  4     1.000000
  2区传感器A                                  2     0.983640
  2区传感器B                                  2     0.946708
  2区传感器C                                  2     0.892405
  3区传感器A                                  3     0.977237
  3区传感器B                                  3     0.949054
  进口温度                                    1     0.903394
  出口温度                                    1     0.889519
  压力变送器A                                  2     0.929356
  压力变送器B                                  2     0.920915
  振动         


NOTE: PROC GVARCLUS data=process_sensors

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.


## 步骤 4 —— 查看输出数据集

来自 `OUTPUT OUT=` 的 `clusters` 数据集中,每个传感器占一行,记录其所属的 `Cluster` 及 `RSq_Own`(该传感器与其聚类分量之间的平方相关系数)。`RSq_Own` 越高,说明该传感器越能被其聚类所代表——是*剔除*、改用该聚类代表测量点的理想候选。我们先按聚类、再按 R 方值排序,使每个聚类中最具代表性的传感器排在该组的最前面。

In [4]:
过程 排序 数据=clusters out=clusters_ranked;
    按照 CLUSTER DESCENDING RSq_Own;
运行;

过程 打印 数据=clusters_ranked 标签;
    变量 Variable CLUSTER RSq_Own;
    标签 Variable = "传感器通道"
          CLUSTER  = "聚类"
          RSq_Own  = "与所属聚类的R方值";
运行;


  Obs            传感器通道      聚类                  与所属聚类的R方值
-----  ---------------  ------  -------------------------
    1  ZONE1_A               1                    0.96867
    2  ZONE1_B               1                     0.9189
    3  TEMP_IN               1                   0.903394
    4  TEMP_OUT              1                   0.889519
    5  ZONE2_A               2                    0.98364
    6  ZONE2_B               2                   0.946708
    7  PRESSURE_A            2                   0.929356
    8  PRESSURE_B            2                   0.920915
    9  ZONE2_C               2                   0.892405
   10  ZONE3_A               3                   0.977237
   11  VIBRATION             3                    0.95916
   12  ZONE3_B               3                   0.949054
   13  ZONE1_C               4                          1




NOTE: PROC SORT data=clusters

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 13 rows from clusters.
NOTE: Wrote clusters_ranked (13 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=clusters_ranked

NOTE: PROC PRINT completed: 13 observations printed, 3 variables


## 结果解读

该划分还原出了产线的大部分物理结构,但有一点需要如实说明:

- **聚类 1** 包含 `zone1_a`(R²=0.969)、`zone1_b`(0.919),以及进/出口温度 `temp_in`(0.903)和 `temp_out`(0.890)——即由1区潜在信号驱动的五个通道中的四个。平均 R 方值为 0.920。
- **聚类 2** 包含全部三个2区通道 `zone2_a`(0.984)、`zone2_b`(0.947)、`zone2_c`(0.892),以及我们与2区驱动信号绑定的两个压力变送器 `pressure_a`(0.929)和 `pressure_b`(0.921)。平均 R 方值为 0.935。
- **聚类 3** 包含 `zone3_a`(0.977)、`zone3_b`(0.949)和 `vibration`(0.959)。平均 R 方值为 0.962。
- **聚类 4** 只有一个通道:`zone1_c`,即第三个1区探头,以 R²=1.000 被单独拆分出来(单变量聚类必然能完美地由自身解释,这是平凡的结果)。尽管 `zone1_c` 也共享1区的驱动信号,但在当前样本量下,该过程判定它与 `zone1_a`/温度组的差异足够大,值得单独成为一类,而不是并入聚类 1。

三个多通道聚类的平均 R 方值均高于 **0.90**,证实单一分量便能解释其成员之间的大部分变异——这正是我们希望合并消除的冗余。唯一的例外——`zone1_c` 独自成类而非并入1区其余通道——提醒我们变量聚类是数据驱动的:若增加循环周期数或提高 `MAXCLUSTERS` 上限,分界可能会发生变化,因此该划分只是工程判断的起点,而非一成不变的定论。

**对产线的行动建议。** 在每个多通道聚类中,保留 `RSq_Own` 最高的传感器(即最能代表该聚类的通道),并将其余通道从常规 SPC 图表中退役或降低优先级——例如聚类 2 保留 `zone2_a`,聚类 3 保留 `zone3_a`。对于 `zone1_c` 这一单变量聚类,应将其视为需要调查的标记,而非自动保留:在决定是否单独监控之前,先确认它究竟携带了真正独特的信息,还是聚类过程的偶然产物。即便暂时搁置这一个通道,这一分析仍将 13 个受监控通道压缩为一个四通道的监控方案,在保留产线信息量的同时降低报警噪声与多重共线性。